# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [1]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 💾 Charger les données

In [2]:
import json
import ast
from pathlib import Path

def load_pyomo_data(input_path="../data/California_data.json"):
    """Charge les données JSON et convertit les clés string en types natifs."""
    input_file = Path(input_path)
    with open(input_file, "r") as f:
        data = json.load(f)

    def _convert_key(key):
        if not isinstance(key, str):
            return key
        if key.startswith("(") and key.endswith(")"):
            try:
                return ast.literal_eval(key)
            except Exception:
                return key
        try:
            return int(key)
        except Exception:
            return key

    # Convertir les dictionnaires de parametres indexes
    params = data.get("params", {})
    for pname, pval in list(params.items()):
        if isinstance(pval, dict):
            params[pname] = {_convert_key(k): v for k, v in pval.items()}

    cartesian = data.get("cartesian_data", {})
    for cname, cval in list(cartesian.items()):
        if isinstance(cval, dict):
            cartesian[cname] = {_convert_key(k): v for k, v in cval.items()}

    data["params"] = params
    data["cartesian_data"] = cartesian
    return data

# Charger les données
data = load_pyomo_data()

## 🔹 Model

In [3]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [4]:
model.VILLES = Set(initialize=data['sets']['VILLES'])

## 🔹 Parameters

In [5]:
model.GainUsine = Param(model.VILLES, initialize=data['params']['GainUsine'], within=NonNegativeReals)
model.GainMagasin = Param(model.VILLES, initialize=data['params']['GainMagasin'], within=NonNegativeReals)
model.CapitalUsine = Param(model.VILLES, initialize=data['params']['CapitalUsine'], within=NonNegativeReals)
model.CapitalMagasin = Param(model.VILLES, initialize=data['params']['CapitalMagasin'], within=NonNegativeReals)
model.Dispo = Param(initialize=data['params']['Dispo'], within=NonNegativeReals)

## 🔹 Variables

In [6]:
model.choix_usine = Var(model.VILLES, domain=Binary)
model.CHOIX_MAGASIN = Var(model.VILLES, domain=Binary)

## 🔹 Constraints

In [7]:
model.c0 = Constraint(expr=sum(model.CapitalMagasin[v]*model.CHOIX_MAGASIN[v]+model.CapitalUsine[v]*model.choix_usine[v] for v in model.VILLES) <= model.Dispo)
model.c1 = Constraint(expr=sum(model.CHOIX_MAGASIN[v] for v in model.VILLES) >= 1)
model.c2 = Constraint(expr=sum(model.choix_usine[v] for v in model.VILLES) >= 1)
model.c_for_0 = ConstraintList()
for v in model.VILLES:
    model.c_for_0.add(model.CHOIX_MAGASIN[v] <= model.choix_usine[v])
# @BIN directive already handled in variable declarations

## 🔹 O

In [8]:
# @BIN directive already handled in variable declarations

## 🔹 Objective

In [9]:
model.obj = Objective(expr=sum(model.GainUsine[v] * model.choix_usine[v] + model.GainMagasin[v] * model.CHOIX_MAGASIN[v] for v in model.VILLES), sense=maximize)

## ⚙️ Résolution du modèle

In [10]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

✅ Solver status: ok
✅ Termination condition: optimal


## 🎯 Valeur de la fonction objective

In [11]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

Objectif: obj
Valeur optimale: 17.0000
Sens: Maximisation


## 📊 Valeurs optimales des variables

In [12]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')

,Variable,Index,Valeur
1,choix_usine,SF,1.0000
2,choix_usine,SD,1.0000
5,CHOIX_MAGASIN,SD,1.0000
